# [2장 2강] - 실습: 벡터공간·선형독립·Rank

**실습 목표**
- 목표 벡터를 다른 벡터들의 선형결합으로 표현하고 span의 의미를 설명할 수 있다.
- 선형독립과 선형종속을 rank로 판별할 수 있다.
- 데이터 행렬의 rank를 계산해 실제로 독립적인 특성이 몇 개인지 해석할 수 있다.
- 영공간의 비자명한 해를 찾아 중복 특성이 존재함을 확인할 수 있다.

- 선형결합과 span → 데이터 행렬 rank → 중복 특성과 영공간 순서로 진행합니다.
- 개인 실습으로 진행하며, rank 값은 항상 "전체 컬럼 수와 비교해서" 해석합니다.

**실습에 필요한 데이터셋/파일**

- 분야: 제조
- 데이터셋: UCI AI4I 2020 Predictive Maintenance
- 사용 방식: `ucimlrepo.fetch_ucirepo(id=601)`
- 출처: https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset
- 사용 목적: 설비 센서 측정값(공기 온도, 공정 온도, 회전 속도, 토크, 마모 시간)으로 데이터 행렬을 만들어 rank와 선형종속을 확인합니다.
- 준비물: Python, NumPy, pandas, scikit-learn, ucimlrepo

In [ ]:
# 최초 1회만 실행 (새 환경일 때)
# !pip -q install ucimlrepo scikit-learn pandas numpy

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


def load_uci(dataset_id):
    """UCI에서 데이터를 불러오고, 실패하면 구조가 비슷한 대체 데이터를 사용합니다."""
    import time
    n_retry, last_err = 4, None
    for attempt in range(1, n_retry + 1):
        try:
            from ucimlrepo import fetch_ucirepo
            ds = fetch_ucirepo(id=dataset_id)
            X, y = ds.data.features.copy(), ds.data.targets.copy()
            if isinstance(y, pd.DataFrame) and y.shape[1] == 1:
                y = y.iloc[:, 0]
            return X, y
        except Exception as e:
            last_err = e
            print(f'[안내] UCI 로드 실패 ({attempt}/{n_retry}회):', e)
            if attempt < n_retry:
                time.sleep(2 * attempt)   # 일시적 네트워크 오류를 대비한 지수 대기

    if last_err is not None:
        print('=' * 72)
        print('[경고] UCI 로드가 최종 실패해 대체 데이터(make_regression)로 진행합니다.')
        print('[경고] 대체 데이터는 컬럼이 서로 독립으로 생성되어 상관계수가 거의 0입니다.')
        print('[경고] 따라서 문제 2-1의 "상관 0.876인데도 rank가 5"라는 실데이터 해석은')
        print('[경고] 이 대체 데이터로는 재현되지 않습니다. 상관계수 숫자 자체가 아니라')
        print('[경고] "상관이 높은 것과 rank가 깎이는 것은 별개"라는 논리에만 집중하세요.')
        print('=' * 72)
        print('[최종 오류]', last_err)
        from sklearn.datasets import make_regression
        Xa, ya = make_regression(n_samples=2000, n_features=5, noise=10,
                                 random_state=RANDOM_STATE)
        cols = ['air_temp', 'process_temp', 'rot_speed', 'torque', 'tool_wear']
        return pd.DataFrame(Xa, columns=cols), pd.Series(ya)


def numeric_frame(X):
    """수치형 컬럼만 남기고 결측값을 중앙값으로 채웁니다."""
    Xn = X.select_dtypes(include='number').copy()
    Xn = Xn.replace([np.inf, -np.inf], np.nan)
    return Xn.fillna(Xn.median(numeric_only=True))


X_raw, y_raw = load_uci(601)   # AI4I 2020 Predictive Maintenance
X_df = numeric_frame(X_raw)
A = StandardScaler().fit_transform(X_df)
print('센서 데이터 행렬 A:', A.shape)
print('컬럼:', list(X_df.columns))

센서 데이터 행렬 A: (10000, 5)
컬럼: ['Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear']


array([[-0.95238944, -0.94735989,  0.06818514,  0.28219976, -1.69598374],
       [-0.90239341, -0.879959  , -0.72947151,  0.63330802, -1.6488517 ],
       [-0.95238944, -1.01476077, -0.22744984,  0.94428963, -1.61743034],
       ...,
       [-0.50242514, -0.94735989,  0.59251888, -0.66077672, -1.35034876],
       [-0.50242514, -0.879959  , -0.72947151,  0.85400464, -1.30321671],
       [-0.50242514, -0.879959  , -0.2162938 ,  0.02137647, -1.22466331]],
      shape=(10000, 5))

## 필수 1 : 어떤 측정값을 다른 측정값들로 만들어 낼 수 있을까
설비 관리팀이 센서를 추가로 달지 검토하고 있습니다. 새 센서가 기존 센서 값들의 조합으로 설명된다면 추가 비용을 들일 이유가 없습니다. 이 판단의 수학적 도구가 선형결합과 span입니다. 작은 벡터 예제로 개념을 먼저 잡고, 다음 단계에서 실제 데이터에 적용합니다.

### 문제 1-1 : 목표 벡터를 선형결합으로 표현하기
1. 기준 벡터 `v1 = [1, 1]`, `v2 = [1, -1]`을 정의합니다. (표준기저 `[1,0]`, `[0,1]`과 달리 계수가 바로 보이지 않으니, 직접 계산해야 합니다.)
2. 목표 벡터 `target = [3, 5]`를 `c1·v1 + c2·v2`로 표현하는 계수 `c1, c2`를 `np.linalg.lstsq`로 구합니다.
3. 구한 계수로 실제 목표 벡터가 복원되는지 확인합니다.
4. `v1`, `v2`가 만드는 span이 무엇인지 한 문장으로 설명합니다.

In [ ]:
# 1. 기준 벡터 `v1 = [1, 1]`, `v2 = [1, -1]`을 정의합니다. (표준기저 `[1,0]`, `[0,1]`과 달리 계수가 바로 보이지 않으니, 직접 계산해야 합니다.)
v1 = np.array([1, 1])
v2 = np.array([1, -1])

# 2. 목표 벡터 `target = [3, 5]`를 `c1·v1 + c2·v2`로 표현하는 계수 `c1, c2`를 `np.linalg.lstsq`로 구합니다.
# c1·v1 + c2·v2 = target
# coeffs, residuals, rank, singular_values = np.linalg.lstsq(A, target, rcond=None)
V = np.column_stack([v1, v2])
target = np.array([3, 5])
c = np.linalg.lstsq(V, target, rcond=None)[0]
c1, c2 = c[0], c[1]
print(f"C1, C2: {c}")

# 3. 구한 계수로 실제 목표 벡터가 복원되는지 확인합니다.
print(f"c1:{c1} * v1:{v1} + c2:{c2} * v2:{v2} = {c1 * v1 + c2 * v2}")
print(f"{np.allclose(target, c1* v1 + c2 * v2)}")

# 4. `v1`, `v2`가 만드는 span이 무엇인지 한 문장으로 설명합니다.
# 모든 선형결합으로 만들어낼 수 있는 벡터들의 집합

C1, C2: [ 4. -1.]
c1:3.9999999999999987 * v1:[1 1] + c2:-1.0 * v2:[ 1 -1] = [3. 5.]
True


### 문제 1-2 : 평행한 벡터들의 span 확인하기
1. 서로 평행한 두 벡터 `w1 = [1, 2]`, `w2 = [2, 4]`(= `2 * w1`)를 정의합니다.
2. 두 벡터를 열로 묶은 행렬의 rank를 계산합니다.
3. 목표 벡터 `[3, 5]`를 이 두 벡터의 선형결합으로 표현해 보고, 복원 결과가 목표와 일치하는지 확인합니다.
4. 목표 벡터 `[2, 4]`(= `w1`의 배수)로 바꿔 다시 시도하고 결과를 비교합니다.
5. 평행한 두 벡터의 span이 왜 평면이 아닌지 한 문장으로 설명합니다.

In [ ]:
# 1. 서로 평행한 두 벡터 `w1 = [1, 2]`, `w2 = [2, 4]`(= `2 * w1`)를 정의합니다.
w1 = np.array([1, 2])
w2 = np.array([2, 4])

# 2. 두 벡터를 열로 묶은 행렬의 rank를 계산합니다.
# coeffs, residuals, rank, singular_values = np.linalg.lstsq(A, target, rcond=None)
W = np.column_stack([w1, w2])
W_rank = np.linalg.matrix_rank(W)
W_rank

# 3. 목표 벡터 `[3, 5]`를 이 두 벡터의 선형결합으로 표현해 보고, 복원 결과가 목표와 일치하는지 확인합니다.
# c1·v1 + c2·v2 = target
c = np.linalg.lstsq(W, target, rcond=None)
print(c)
print(f"c값: {c[0]}")
print(f"c1*w1 + c2*w2: {c[0][0] * w1 + c[0][1] * w2}")

# 4. 목표 벡터 `[2, 4]`(= `w1`의 배수)로 바꿔 다시 시도하고 결과를 비교합니다.
target_test = np.array([2, 4])
c_test = np.linalg.lstsq(W, target_test, rcond=None)
print(f"c_test값: {c[0]}")
print(f"c1*w1 + c2*w2: {c_test[0][0] * w1 + c_test[0][1] * w2}")

# 5. 평행한 두 벡터의 span이 왜 평면이 아닌지 한 문장으로 설명합니다.
# 평행한 두 벡터는 원점을 기준으로 한 직선위에 놓이며, 어떠한 선형결합으로도 직선을 벗어나지 못하므로 span은 직선일 뿐 평면이 되지 못한다.

(array([0.52, 1.04]), array([], dtype=float64), np.int32(1), array([5.00000000e+00, 1.04061363e-16]))
c값: [0.52 1.04]
c1*w1 + c2*w2: [2.6 5.2]
c_test값: [0.52 1.04]
c1*w1 + c2*w2: [2. 4.]


## 필수 2 : 센서가 5개인데 진짜 정보는 몇 개일까
설비 데이터에 센서 컬럼이 여러 개 있지만, 공기 온도와 공정 온도처럼 서로 강하게 연동되는 항목이 있습니다. 컬럼 수가 곧 정보량은 아니므로, 데이터 행렬의 rank를 계산해 실제로 독립적인 특성이 몇 개인지 확인합니다.

## 문제 2-1 : 데이터 행렬의 rank 계산하기
1. `A`의 shape과 `np.linalg.matrix_rank(A)`를 계산합니다.
2. 가능한 최대 rank(`min(A.shape)`)와 비교합니다.
3. 컬럼 간 상관계수 행렬을 출력하고, 상관계수의 절댓값이 가장 큰 컬럼 쌍 두 개를 찾아 적습니다. (공기 온도-공정 온도 ≈ 0.876, 회전 속도-토크 ≈ -0.875)
4. 상관계수가 0.876이나 되는 컬럼 쌍이 있는데도 rank가 컬럼 수와 동일하게 5로 나오는 이유를 설명합니다. 그리고 두 컬럼이 **정확하게 선형종속**이 되어 rank가 실제로 깎이려면 상관계수가 얼마여야 하는지 답합니다. (이 답은 바로 다음 문제 2-2에서 상관 1.0인 컬럼을 직접 만들어 확인합니다.)

In [46]:
# 1. `A`의 shape과 `np.linalg.matrix_rank(A)`를 계산합니다.
print(A.shape) #(10000, 5)
print(np.linalg.matrix_rank(A)) #5

# 2. 가능한 최대 rank(`min(A.shape)`)와 비교합니다.
print("가능한 최대 rank:", min(A.shape))

# 3. 컬럼 간 상관계수 행렬을 출력하고, 상관계수의 절댓값이 가장 큰 컬럼 쌍 두 개를 찾아 적습니다. (공기 온도-공정 온도 ≈ 0.876, 회전 속도-토크 ≈ -0.875)
# corr = np.corrcoef(A, rowvar=False)
# np.set_printoptions(linewidth=np.inf)

# cols = list(X_df.columns)
# print(cols)
# print(corr)

corr = pd.DataFrame(A, columns=X_df.columns).corr()
display(corr.round(3))

pairs = (corr.where(~np.eye(len(corr), dtype=bool))
             .stack()
             .drop_duplicates()
             .sort_values(key=np.abs, ascending=False))
print('\n상관계수 |r| 상위 컬럼 쌍')
print(pairs.head(3).round(3))
print(f'\n최대 |r| = {pairs.abs().max():.3f}  ->  1.0이 아니므로 정확한 선형종속은 아님')

# 4. 상관계수가 0.876이나 되는 컬럼 쌍이 있는데도 rank가 컬럼 수와 동일하게 5로 나오는 이유를 설명합니다. 그리고 두 컬럼이 **정확하게 선형종속**이 되어 rank가 실제로 깎이려면 상관계수가 얼마여야 하는지 답합니다. (이 답은 바로 다음 문제 2-2에서 상관 1.0인 컬럼을 직접 만들어 확인합니다.)
# rank가 깍이려면 한 컬럼이 다른 컬럼(들)으로 완벽하게 표현이 되어야 하는데, 그 만큼 겹치지 않아 선형독립이다.
# 상관계수가 +1, -1 이어야 한다.

(10000, 5)
5
가능한 최대 rank: 5


,Air temperature,Process temperature,Rotational speed,Torque,Tool wear
Air temperature,1.000,0.876,0.023,-0.014,0.014
Process temperature,0.876,1.000,0.019,-0.014,0.013
Rotational speed,0.023,0.019,1.000,-0.875,0.000
Torque,-0.014,-0.014,-0.875,1.000,-0.003
Tool wear,0.014,0.013,0.000,-0.003,1.000



상관계수 |r| 상위 컬럼 쌍
Air temperature   Process temperature    0.876
Rotational speed  Torque                -0.875
Air temperature   Rotational speed       0.023
dtype: float64

최대 |r| = 0.876  ->  1.0이 아니므로 정확한 선형종속은 아님


### 문제 2-2 : 선형종속 컬럼을 추가해도 rank가 늘지 않음 확인하기
1. 기존 행렬 `A`에 두 개의 컬럼을 추가한 `A_dep`을 만듭니다.
    - `A[:, 0] * 2` (첫 컬럼의 배수)
    - `A[:, 1] + A[:, 2]` (두 컬럼의 합)
2. `A`와 `A_dep`의 shape과 rank를 각각 계산해 비교합니다.
3. 추가로 **난수 컬럼** 하나를 붙인 `A_rand`의 rank도 계산해 비교합니다.
4. "컬럼이 늘어도 rank가 늘지 않는" 경우와 "늘어나는" 경우의 차이를 한 문장으로 설명합니다.

In [ ]:
# 1. 기존 행렬 `A`에 두 개의 컬럼을 추가한 `A_dep`을 만듭니다.
#     - `A[:, 0] * 2` (첫 컬럼의 배수)
#     - `A[:, 1] + A[:, 2]` (두 컬럼의 합)
A_dep = np.column_stack([
    A,
    A[:, 0] * 2, 
    A[:, 1] + A[:, 2]
    ])
# print(A_dep)

# 2. `A`와 `A_dep`의 shape과 rank를 각각 계산해 비교합니다.
print(f"A.shape: {A.shape}, A rank: {np.linalg.matrix_rank(A)}")
print(f"A_dep.shape: {A_dep.shape}, A_dep rank: {np.linalg.matrix_rank(A_dep)}")

# 3. 추가로 **난수 컬럼** 하나를 붙인 `A_rand`의 rank도 계산해 비교합니다.
A_rand = np.column_stack([
    A,
    np.random.randn(10000)
])
print(f"A_rand.shape: {A_rand.shape}, A_rand rank: {np.linalg.matrix_rank(A_rand)}")

# 4. "컬럼이 늘어도 rank가 늘지 않는" 경우와 "늘어나는" 경우의 차이를 한 문장으로 설명합니다.
# rank가 늘지 않는 경우는 새 컬럼이 기존 컬럼(들)의 선형결합으로 만들어지는 경우이며, 늘어나는 경우는 새 컬럼이 기존것들로 만들 수 없는 독립적인 방향일 때이다.

A.shape: (10000, 5), A rank: 5
A_dep.shape: (10000, 7), A_dep rank: 5
A_rand.shape: (10000, 6), A_rand rank: 6


## 심화 1 : 중복 특성이 남긴 흔적, 영공간
rank가 컬럼 수보다 작다는 것은 알았지만, 어떤 컬럼이 어떤 조합으로 중복인지까지 알면 제거 대상을 특정할 수 있습니다. Ax = 0을 만족하는 0이 아닌 벡터 x(영공간의 원소)가 바로 그 중복 관계를 알려 줍니다.

### 문제 3-1 : 영공간의 비자명한 해로 중복 관계 확인하기
1. `A_dep`에서 "첫 컬럼의 2배가 마지막에서 두 번째 컬럼"이라는 관계를 계수 벡터 `c1`으로 표현합니다.
2. `A_dep @ c1`이 0에 가까운지 확인합니다. (영공간의 원소)
3. `A[:, 1] + A[:, 2]`가 마지막 컬럼이라는 관계도 계수 벡터 `c2`로 표현하고 같은 방식으로 확인합니다.
4. `A_dep`의 컬럼 수, rank, 영공간의 차원(= 컬럼 수 − rank)을 계산해 세 값의 관계를 확인합니다.
5. 비교를 위해 `A`(중복 없음)에서는 `Ax = 0`의 해가 `x = 0`뿐인지 rank로 확인합니다.
6. 영공간의 차원이 0보다 클 때 데이터·모델 관점에서 어떤 의미인지 2~3문장으로 작성합니다.

In [ ]:
# 1. `A_dep`에서 "첫 컬럼의 2배가 마지막에서 두 번째 컬럼"이라는 관계를 계수 벡터 `c1`으로 표현합니다.
print(A_dep.shape) #(10000, 7)
# A_dep = np.column_stack([
#     A,
#     A[:, 0] * 2, 
#     A[:, 1] + A[:, 2]
#     ])
c1 = np.array([2, 0, 0, 0, 0, -1, 0])

# 2. `A_dep @ c1`이 0에 가까운지 확인합니다. (영공간의 원소)
print(A_dep @ c1)

# 3. `A[:, 1] + A[:, 2]`가 마지막 컬럼이라는 관계도 계수 벡터 `c2`로 표현하고 같은 방식으로 확인합니다.
c2 = np.array([0, 1, 1, 0, 0, 0, -1])
print(A_dep @ c2)

# 4. `A_dep`의 컬럼 수, rank, 영공간의 차원(= 컬럼 수 − rank)을 계산해 세 값의 관계를 확인합니다.
print("A_dep의 컬럼 수: ", A_dep.shape[1])
# rank는 다른 컬럼들의 선형결합으로 만들어지지 않는 독립적인 컬럼의 수
print("A_dep의 rank: ", np.linalg.matrix_rank(A_dep))
print("A_dep 영공간의 차원: ", A_dep.shape[1] - np.linalg.matrix_rank(A_dep))

# 5. 비교를 위해 `A`(중복 없음)에서는 `Ax = 0`의 해가 `x = 0`뿐인지 rank로 확인합니다.
print("A의 컬럼 수: ", A.shape[1])
print("A의 rank: ", np.linalg.matrix_rank(A))
print("A 영공간의 차원: ", A.shape[1] - np.linalg.matrix_rank(A))

# 6. 영공간의 차원이 0보다 클 때 데이터·모델 관점에서 어떤 의미인지 2~3문장으로 작성합니다.
# 어떤 컬럼이 다른 컬럼들의 선형결합으로 만들어지는 중복 특성이라는 뜻이다.
# 새로운 정보가 없는 컬럼이므로 가중치의 해가 유일하게 정해지지 않아 학습이 불안정하고 특성 중요도 해석이 어려워진다.

(10000, 7)
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
A_dep의 컬럼 수:  7
A_dep의 rank:  5
A_dep 영공간의 차원:  2
A의 컬럼 수:  5
A의 rank:  5
A 영공간의 차원:  0
